# 06b - Memodelkan Kelas yang Tidak Seimbang
Materi Modul 6: pengantar kelas tidak seimbang, upsampling dan downsampling, pembobotan dan pengambilan sampel bertingkat, oversampling acak dan sintetis (SMOTE), metode tetangga terdekat, blagging.

Dataset: `kelas_tidak_seimbang.csv` (5000 baris, 10 fitur, kelas positif hanya ~5%, misal transaksi penipuan).

**Aturan emas:** resampling hanya boleh dilakukan pada **data train**. Data test dibiarkan asli. Pakai `imblearn.pipeline.Pipeline` agar resampling terjadi di dalam tiap fold CV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, classification_report)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, NearMiss, EditedNearestNeighbours
from imblearn.ensemble import BalancedBaggingClassifier

df = pd.read_csv('../datasets/kelas_tidak_seimbang.csv')
X, y = df.drop(columns='target'), df['target']
print(y.value_counts())
print('Proporsi kelas positif:', round(y.mean(), 4))

## 1. Split bertingkat (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print('Positif di train:', y_train.mean().round(4), '| di test:', y_test.mean().round(4))

## 2. Kenapa akurasi menipu?
Model yang selalu menebak 'negatif' sudah akurasi ~95%, tapi tidak berguna.

In [ ]:
pred_nol = np.zeros(len(y_test), dtype=int)
print('Akurasi "selalu negatif":', round((pred_nol == y_test).mean(), 4))
print('Recall                  :', recall_score(y_test, pred_nol))

In [ ]:
def evaluasi(nama, model, Xtr=X_train, ytr=y_train):
    model.fit(Xtr, ytr)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    print(f'{nama:32s} | presisi={precision_score(y_test, pred, zero_division=0):.3f} | recall={recall_score(y_test, pred):.3f} | '
          f'F1={f1_score(y_test, pred):.3f} | ROC-AUC={roc_auc_score(y_test, proba):.3f} | PR-AUC={average_precision_score(y_test, proba):.3f}')

evaluasi('Baseline (tanpa penanganan)', ImbPipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]))

## 3. Pembobotan kelas (`class_weight`)

In [ ]:
evaluasi('class_weight=balanced', ImbPipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000, class_weight='balanced'))]))
evaluasi('class_weight manual 1:10', ImbPipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000, class_weight={0: 1, 1: 10}))]))

## 4. Upsampling dan downsampling manual (pakai `sklearn.utils.resample`)

In [ ]:
train = pd.concat([X_train, y_train], axis=1)
mayor = train[train['target'] == 0]
minor = train[train['target'] == 1]

# Upsampling: gandakan kelas minoritas (dengan pengembalian)
minor_up = resample(minor, replace=True, n_samples=len(mayor), random_state=42)
train_up = pd.concat([mayor, minor_up])

# Downsampling: kurangi kelas mayoritas
mayor_down = resample(mayor, replace=False, n_samples=len(minor), random_state=42)
train_down = pd.concat([mayor_down, minor])

print('Asli        :', train['target'].value_counts().to_dict())
print('Upsampling  :', train_up['target'].value_counts().to_dict())
print('Downsampling:', train_down['target'].value_counts().to_dict())

In [ ]:
model_lr = lambda: ImbPipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))])
evaluasi('Upsampling manual', model_lr(), train_up.drop(columns='target'), train_up['target'])
evaluasi('Downsampling manual', model_lr(), train_down.drop(columns='target'), train_down['target'])

## 5. Oversampling acak dan sintetis (SMOTE)
SMOTE membuat titik **baru** di antara tetangga kelas minoritas, bukan sekadar menyalin.

In [ ]:
evaluasi('Random oversampling', ImbPipeline([('r', RandomOverSampler(random_state=42)), ('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]))
evaluasi('SMOTE', ImbPipeline([('r', SMOTE(random_state=42)), ('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]))
evaluasi('Random undersampling', ImbPipeline([('r', RandomUnderSampler(random_state=42)), ('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]))

## 6. Metode tetangga terdekat (Tomek Links, ENN, NearMiss)
Membersihkan / memilih sampel kelas mayoritas berdasarkan tetangganya.

In [ ]:
sc = StandardScaler().fit(X_train)
Xs = sc.transform(X_train)
for nama, s in [('Tomek Links', TomekLinks()), ('ENN', EditedNearestNeighbours()), ('NearMiss', NearMiss())]:
    Xr, yr = s.fit_resample(Xs, y_train)
    print(f'{nama:12s} | sebelum: {np.bincount(y_train)} | sesudah: {np.bincount(yr)}')

print()
evaluasi('Tomek Links', ImbPipeline([('s', StandardScaler()), ('r', TomekLinks()), ('m', LogisticRegression(max_iter=1000))]))
evaluasi('ENN (Edited Nearest Neighbours)', ImbPipeline([('s', StandardScaler()), ('r', EditedNearestNeighbours()), ('m', LogisticRegression(max_iter=1000))]))

## 7. Blagging (Balanced Bagging)
Bagging di mana tiap bootstrap sudah diseimbangkan lebih dulu.

In [ ]:
evaluasi('Random Forest biasa', RandomForestClassifier(n_estimators=200, random_state=42))
evaluasi('Random Forest balanced', RandomForestClassifier(n_estimators=200, class_weight='balanced_subsample', random_state=42))
evaluasi('Blagging (BalancedBagging)', BalancedBaggingClassifier(n_estimators=50, random_state=42))

## 8. Validasi silang yang benar (resampling di dalam fold)

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
kandidat = {
    'Baseline': ImbPipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000))]),
    'SMOTE': ImbPipeline([('s', StandardScaler()), ('r', SMOTE(random_state=42)), ('m', LogisticRegression(max_iter=1000))]),
    'class_weight': ImbPipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
}
for nama, m in kandidat.items():
    h = cross_validate(m, X_train, y_train, cv=kf, scoring=['recall', 'precision', 'f1', 'average_precision'])
    print(f'{nama:13s} | recall={h["test_recall"].mean():.3f} | presisi={h["test_precision"].mean():.3f} | F1={h["test_f1"].mean():.3f} | PR-AUC={h["test_average_precision"].mean():.3f}')

## 9. Ringkasan metode

| Metode | Cara kerja | Catatan |
|---|---|---|
| `class_weight` | Beri bobot lebih pada kelas minoritas | Tanpa mengubah data |
| Upsampling / random oversampling | Gandakan sampel minoritas | Risiko overfit (duplikat) |
| Downsampling / undersampling | Buang sampel mayoritas | Bisa membuang informasi |
| SMOTE | Buat sampel minoritas sintetis | Bisa menciptakan noise |
| Tomek / ENN / NearMiss | Pilih/bersihkan mayoritas berdasar tetangga | Peka terhadap skala fitur |
| Blagging | Bagging dengan bootstrap seimbang | Cocok dengan model berbasis pohon |

## Catatan
- (kenapa akurasi tidak cocok, kapan pakai PR-AUC, kenapa resampling hanya di train, dll.)